# BERT fine-tuning

This notebook fine-tunes `bert-base-uncased` on our training data. It is the transformer
baseline the professor named, and it is the first model in the benchmark where the whole
network is trained rather than used as a fixed feature extractor.

**Architecture vs specific model.** BERT is the architecture: a 12-layer transformer encoder
trained with masked language modelling. `bert-base-uncased` is the specific checkpoint we
load: about 110M parameters, English only, lowercased input.

**What fine-tuning means here.** In notebook 03 the sentence transformer was *frozen* — it
only produced vectors, and the only thing trained was the SVM on top. Here every weight in
BERT is updated by our 2,104 training messages, plus a new 14-way classification head.
That is the difference the benchmark is trying to measure.

Both test sets are scored once at the end: the synthetic test set and the real MINDS-14
messages. The score to beat on MINDS-14 is **0.951** (Jev).
## 0. Colab setup

This notebook needs a GPU. In Colab: **Runtime → Change runtime type → T4 GPU**.

The cell below clones the repository. The token is typed at runtime with `getpass`, so it is
never stored in the notebook file and never appears in the output. Skip this cell if you are
running locally from `notebooks/`.

In [ ]:
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    import getpass, os, subprocess

    REPO = "IbragimAliyev/Intent_Classification_For_Banking"
    token = getpass.getpass("GitHub token (input hidden): ")
    result = subprocess.run(
        ["git", "clone", f"https://{token}@github.com/{REPO}.git", "/content/repo"],
        capture_output=True, text=True,
    )
    print("clone ok" if result.returncode == 0
          else result.stderr.replace(token, "***"))
    del token

    os.chdir("/content/repo/notebooks")
    !pip -q install "transformers>=4.46" "accelerate>=0.26"

print("working dir:", __import__("os").getcwd())

## 1. Imports

`src/finetune.py` holds the training code. It is shared with notebook 06 (XLM-RoBERTa) so
both models are trained by exactly the same function — same seed, same batch size, same
early-stopping rule. If the two were trained by separate code, a difference between them
could come from the training setup rather than from the model.

In [ ]:
import os
import pandas as pd
import torch

import sys
sys.path.insert(0, '..')


from src.evaluate import evaluate
from src.finetune import finetune, predict_intents, best_epoch_score
SEED = 42
MODEL_NAME = "bert-base-uncased"

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — stop and switch runtime to T4")

## 2. Load the data

The same split as every other notebook, plus the real MINDS-14 messages Mahmut prepared.
MINDS-14 is a test set only — we never train on it.
- **train**: fine-tunes the model
- **validation**: picks the learning rate and the epoch
- **synthetic test** and **minds14**: scored once each, at the very end

In [ ]:
train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/val.csv")
test_df = pd.read_csv("../data/processed/test.csv")
minds_df = pd.read_csv("../data/processed/minds14_en.csv")

print("train:  ", train_df.shape)
print("val:    ", val_df.shape)
print("test:   ", test_df.shape)
print("minds14:", minds_df.shape)
print()
print("intents match:", sorted(train_df['intent'].unique()) == sorted(minds_df['intent'].unique()))

## 3. Check the message lengths

`max_length` decides where BERT cuts a message off. Too small and we lose words; too large
and we waste memory and time padding. We check the real lengths before fixing the value.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
for name, frame in [("train", train_df), ("synthetic test", test_df), ("minds14", minds_df)]:
    lengths = [len(tok.encode(str(t))) for t in frame["text"]]
    over = sum(1 for n in lengths if n > 64)
    print(f"{name:15} max {max(lengths):3d}   mean {sum(lengths)/len(lengths):5.1f}   over 64: {over} ({100*over/len(lengths):.1f}%)")

## 4. Tune the learning rate on validation

Learning rate is the setting that matters most when fine-tuning a transformer. Too high and
the pretrained weights get destroyed in the first few steps; too low and the model barely
moves. `2e-5`, `3e-5` and `5e-5` are the standard range for BERT.

Each run trains up to 5 epochs and evaluates on validation after every epoch. The epoch with
the best validation macro-F1 is kept (`load_best_model_at_end`), so a run that starts to
overfit in later epochs is not penalised for it — we simply keep its best point.

The test sets are not touched anywhere in this section.

In [ ]:
learning_rates = [2e-5, 3e-5, 5e-5]
results = []
trainers = {}

for lr in learning_rates:
    print(f"\n=== learning rate {lr} ===")
    trainer, labels = finetune(
        MODEL_NAME, train_df, val_df,
        learning_rate=lr, num_epochs=5, batch_size=16, seed=SEED,
    )
    epoch, score = best_epoch_score(trainer)
    results.append({"learning_rate": lr, "best_epoch": epoch, "val_macro_f1": round(score, 4)})
    trainers[lr] = trainer

tuning_df = pd.DataFrame(results)
print()
print(tuning_df)

best_lr = tuning_df.loc[tuning_df["val_macro_f1"].idxmax(), "learning_rate"]
print("\nBest learning rate:", best_lr)

## 5. The final model

The run with the best validation macro-F1 already has its best epoch loaded back in, so that
trainer *is* the final model. Retraining it with the same learning rate and the same seed
would reproduce it exactly, so we reuse it instead of spending another five epochs of GPU
time on an identical result.

In [ ]:
final_trainer = trainers[best_lr]
best_row = tuning_df.loc[tuning_df["val_macro_f1"].idxmax()]
print(f"learning rate {best_row['learning_rate']}, epoch {best_row['best_epoch']}, "
      f"validation macro-F1 {best_row['val_macro_f1']}")

## 6. Evaluate on both test sets, once

Scored with the shared `evaluate()` so the rows land in `results/metrics.csv` next to every
other model. **Run this cell once.** Each run appends a row; a second run creates duplicates
that somebody has to delete by hand.

In [ ]:
test_preds = predict_intents(final_trainer, test_df["text"], labels)
minds_preds = predict_intents(final_trainer, minds_df["text"], labels)

evaluate(test_df["intent"], test_preds, "bert_base", "synthetic_test", out_dir="../results")
evaluate(minds_df["intent"], minds_preds, "bert_base", "minds14", out_dir="../results")

## 7. Save the predictions

`models/` is not in git and the fine-tuned weights live on Colab, so the predictions are the
only thing that survives. Mahmut's bootstrap confidence intervals and the final benchmark
table are both built from these files, not from the model.

In [ ]:
os.makedirs("../results/predictions", exist_ok=True)

pd.DataFrame({"text": test_df["text"], "intent": test_df["intent"], "pred": test_preds}).to_csv(
    "../results/predictions/bert_base_synthetic_test.csv", index=False
)
pd.DataFrame({"text": minds_df["text"], "intent": minds_df["intent"], "pred": minds_preds}).to_csv(
    "../results/predictions/bert_base_minds14.csv", index=False
)
print("saved both prediction files")

## 8. Compare with the models already in the benchmark

The numbers below come straight from `results/metrics.csv`, so they include the rows this
notebook just added.

In [ ]:
metrics = pd.read_csv("../results/metrics.csv")
for dataset in ["synthetic_test", "minds14"]:
    print(f"\n=== {dataset} ===")
    print(metrics[metrics["dataset"] == dataset]
          .sort_values("macro_f1", ascending=False)
          .to_string(index=False))

## 9. Conclusion

- Best learning rate: ... , best epoch: ...
- Validation macro-F1: ...
- synthetic_test: accuracy ... | macro-F1 ...
- minds14: accuracy ... | macro-F1 ... (score to beat was 0.951 from Jev)
- Fine-tuned BERT vs the frozen encoder in notebook 03 (sbert_svm: 0.9866 synthetic, 0.9170 minds14): ...
- Drop from synthetic to real data: ... (jev drops about 4 points, the trained models about 7)
- Weakest intents on minds14, from the confusion matrix: ...
